#### **📘 Source Freshness Checks in dbt**

In **dbt, source freshness checks** answer one simple but critical question:

> **“How recent is my raw data?”**

They help you detect **broken or delayed ingestion pipelines** *before* bad or stale data flows into transformations, marts, and dashboards.

--------------

#### **1️⃣ What is Source Freshness? (Definition)**

**✅ Definition**

**Source freshness** is a dbt feature that checks **when a source table was last updated** and compares it against **acceptable time limits** that you define.

In simple terms:

> dbt looks at the *latest timestamp* in a source table and asks
“Is this data fresh enough to trust?”

-----------

#### **2️⃣ Why Source Freshness Exists (The Real Problem)**

**Without freshness checks**

- Raw ingestion silently fails

- dbt transformations still run

- Dashboards show **old data**

- Business users lose trust

You only find out **after damage is done**.


**With freshness checks**

- dbt detects stale data early

- Fails or warns **before downstream models**

- Makes ingestion failures visible immediately

----------------

**3️⃣ How dbt Measures Freshness (Very Important)**

dbt **does NOT magically know** when data was loaded.

You must tell dbt:

- **Which timestamp column to check**

- **How old is “too old”**

------------

**4️⃣ Core Components of Freshness Checks**

**🔹 1. loaded_at_field**

This is the column dbt uses to determine freshness.

Example:

In [ ]:
loaded_at_field: review_date

dbt runs:

In [ ]:
SELECT MAX(review_date) FROM raw.raw_reviews

--------

**🔹 2. warn_after**

Defines when dbt should **warn** (yellow flag).

In [ ]:
warn_after:
  count: 6
  period: hour

Meaning:

> If data is older than **6 hours**, raise a warning.

-------

**🔹 3. error_after**

Defines when dbt should **fail** (red flag).

In [ ]:
error_after:
  count: 12
  period: hour

Meaning:

If data is older than **12 hours**, fail the job.

------------

#### **5️⃣ Full Example (Airbnb – Reviews)**

In [ ]:
version: 2

sources:
  - name: airbnb
    schema: raw
    tables:
      - name: reviews
        identifier: raw_reviews
        loaded_at_field: review_date
        freshness:
          warn_after:
            count: 6
            period: hour
          error_after:
            count: 12
            period: hour

------------

#### **6️⃣ What Happens When You Run Freshness Checks**

Run this command:

In [ ]:
dbt source freshness

dbt does **NOT** run models.

It only checks timestamps.

---------------

**Case A: Data is fresh ✅**

- Latest `review_date` = 30 minutes ago

- No warnings

- No errors

Output:

In [ ]:
PASS source.airbnb.reviews

------

**Case B: Data is slightly stale ⚠️**

- Latest `review_date` = 7 hours ago

- `warn_after` = 6 hours

Output:

In [ ]:
WARN source.airbnb.reviews
Freshness exceeded warn threshold

👉 Pipeline continues, but you are alerted.

-----------------


**Case C: Data is very stale ❌**

- Latest `review_date` = 14 hours ago

- `error_after` = 12 hours

Output:

In [ ]:
ERROR source.airbnb.reviews
Freshness exceeded error threshold

👉 dbt **fails.**

👉 Downstream models should not run.

------------

#### **7️⃣ What Freshness Checks Protect You From**

| Problem              | Without Freshness | With Freshness |
| -------------------- | ----------------- | -------------- |
| Ingestion job failed | Silent            | Detected       |
| Stale dashboards     | Likely            | Prevented      |
| Debug time           | Hours             | Minutes        |
| Root cause clarity   | Low               | High           |


---------------


#### **8️⃣ Freshness vs Tests (Very Important Difference)**

| Feature  | Freshness              | Tests         |
| -------- | ---------------------- | ------------- |
| Checks   | Time-based             | Data quality  |
| Detects  | Stale data             | Bad data      |
| Uses     | Timestamp column       | Column values |
| Runs via | `dbt source freshness` | `dbt test`    |


👉 Freshness answers “Is data recent?”

👉 Tests answer “Is data valid?”



-----------

#### **9️⃣ Common Mistakes (Avoid These)**

❌ Using a non-timestamp column

❌ Using business dates instead of load timestamps

❌ Setting unrealistically strict thresholds

❌ Expecting freshness to validate data quality

❌ Forgetting to run dbt source freshness in jobs

--------------------

#### **🔑 Best Practices (Production-Grade)**

- Use **ingestion timestamps** (`_loaded_at`, `_synced_at`)

- Add freshness only to **critical sources**

- Align thresholds with ingestion SLA

- Fail (`error_after`) for business-critical pipelines

- Monitor warnings regularly

--------------

#### **🧠 One-Line Memory Lock**

> **Source freshness checks ensure dbt never trusts old data.**